In [4]:
from astroquery.simbad import Simbad
import pandas as pd
import numpy as np

In [5]:
class Gaia_info:
    def __init__(self, timeout=10000):
        self.gaia_simbad = Simbad()
        self.gaia_simbad.add_votable_fields('ids')
        self.gaia_simbad.TIMEOUT = timeout

    def get_gaia_info(self, targets):
        xmatch = self.simbad_xmatch(targets)
        corrected_xmatch = self.rm_duplicates(xmatch)
        return self.join_xmatch(targets, corrected_xmatch)

    def simbad_xmatch(self, targets):
        simbad_res = self.gaia_simbad.query_objects(targets)
        if simbad_res is None:
            return pd.DataFrame(columns=['Gaia_ID'])

        df = simbad_res.to_pandas()
        if 'Gaia_ID' not in df.columns:
            df['Gaia_ID'] = self._extract_gaia_id(df)

        missing = df['Gaia_ID'].isna() if 'Gaia_ID' in df.columns else pd.Series([True] * len(df))
        if missing.any():
            for i in np.where(missing)[0]:
                row = df.iloc[i]
                main_id = self._get_first_existing(row, ['MAIN_ID', 'main_id', 'matched_id', 'user_specified_id'])
                if pd.notna(main_id):
                    df.at[i, 'Gaia_ID'] = self._query_gaia_id_for_object(main_id)

        return df

    def _get_first_existing(self, row, keys):
        for key in keys:
            if key in row.index and pd.notna(row[key]):
                return row[key]
        return pd.NA

    def _extract_gaia_id(self, df):
        for key in ['ID_gaia', 'ID_GAIA', 'gaia_id', 'Gaia_id']:
            if key in df.columns:
                return df[key]

        ids_col = None
        for key in ['IDS', 'ids', 'ID_LIST', 'id_list']:
            if key in df.columns:
                ids_col = key
                break
        if ids_col is None:
            return pd.Series([pd.NA] * len(df))

        extracted = df[ids_col].astype(str).str.extract(
            r'Gaia\s+(?:DR\d+|EDR\d+)?\s*(\d+)',
            expand=False
        )
        return extracted

    def _query_gaia_id_for_object(self, object_name):
        try:
            ids_table = self.gaia_simbad.query_objectids(str(object_name))
            if ids_table is None:
                return pd.NA

            ids_df = ids_table.to_pandas()
            id_col = 'ID' if 'ID' in ids_df.columns else ('id' if 'id' in ids_df.columns else None)
            if id_col is None:
                return pd.NA

            matches = ids_df[id_col].astype(str).str.extract(
                r'Gaia\s+(?:DR\d+|EDR\d+)?\s*(\d+)',
                expand=False
            ).dropna()
            return matches.iloc[0] if len(matches) > 0 else pd.NA
        except Exception:
            return pd.NA

    def rm_duplicates(self, dataframe):
        dup_key = None
        for key in ['SCRIPT_NUMBER_ID', 'object_number_id']:
            if key in dataframe.columns:
                dup_key = key
                break
        if dup_key is None:
            return dataframe.reset_index(drop=True)
        return dataframe.drop_duplicates(subset=dup_key, keep='first').reset_index(drop=True)

    def join_xmatch(self, targets, df):
        target_df = pd.DataFrame({'Target': targets})

        join_key = None
        for key in ['SCRIPT_NUMBER_ID', 'object_number_id']:
            if key in df.columns:
                join_key = key
                break

        if join_key is not None:
            df = df.copy()
            df[join_key] = df[join_key].astype(int) - 1
            out = target_df.join(df.set_index(join_key))
        else:
            out = target_df.join(df.reset_index(drop=True))

        out.rename(
            columns={'MAIN_ID': 'Simbad_ID', 'main_id': 'Simbad_ID', 'matched_id': 'Simbad_ID'},
            inplace=True
        )
        if 'Simbad_ID' not in out.columns:
            out['Simbad_ID'] = pd.NA
        if 'Gaia_ID' not in out.columns:
            out['Gaia_ID'] = pd.NA
        return out

In [3]:
groups = pd.read_pickle("/home/msp25gd/Downloads/res/meta/groups.pkl")
groups

,New Groups,OBJECT,Sanitised,Reduced,DEC,EXPTIME,MJD-OBS,MJD-END,WAVELMIN,WAVELMAX,SPEC_BIN,SPEC_RES
0,0,$\alpha$-Cru,$\alpha$ Cru,$\alpha$cru,-63.09899,5.0078,53747.372964,53747.375327,328.195459,456.300129,0.001358,65030.0
1,1,$\gamma^2$-Vel,$\gamma^2$ Vel,$\gamma^2$vel,-47.33665,5.0079,53670.279645,53670.281882,328.192145,456.299040,0.001357,65030.0
2,2,$\theta$-Car,$\theta$ Car,$\theta$car,-64.39407,10.0084,53686.311039,53686.313337,328.196918,456.298107,0.001356,65030.0
3,3,$\zeta$-Pup,$\zeta$ Pup,$\zeta$pup,-40.00284,5.0081,53670.273791,53670.276006,328.192145,456.299040,0.001357,65030.0
4,4,0003+1713,0003+1713,0003+1713,17.22642,1800.0037,59487.233078,59487.254500,328.196045,456.297607,0.002710,19540.0
...,...,...,...,...,...,...,...,...,...,...,...,...
12822,12822,zetaPup,zetaPup,zetapup,-40.00273,2.0022,52005.963535,52005.963559,373.122100,499.931900,0.001500,65030.0
12823,12823,zeta-Tau,zeta Tau,zetatau,21.14288,3.0043,53333.271293,53333.272410,328.190157,456.293935,0.001352,58640.0
12824,12824,zet Per,zet Per,zetper,31.88373,127.9981,60597.320634,60597.330454,373.220080,499.982989,0.001477,71050.0
12825,12825,Z-Sct,Z Sct,zsct,-5.82086,150.0015,54678.090211,54678.091947,373.217100,499.984500,0.001500,36840.0
